<a href="https://colab.research.google.com/github/amigli/Q-Bert_RL/blob/main/Notebook/A2C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training e Test con A2C
In questo notebook è presente il training di Q*Bert sfruttando l'algoritmo A2C

## Download Repository

In [1]:
from google.colab import userdata

In [2]:
!git clone https://{userdata.get('TokenGithub')}"@github.com/amigli/Q-Bert_RL.git"

Cloning into 'Q-Bert_RL'...
remote: Enumerating objects: 598, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 598 (delta 40), reused 27 (delta 17), pack-reused 528 (from 1)
Receiving objects: 100% (598/598), 10.46 MiB | 9.18 MiB/s, done.
Resolving deltas: 100% (377/377), done.


In [3]:
%cd Q-Bert_RL/

/content/Q-Bert_RL


## Installazione dei requirements

In [4]:
!pip install gymnasium

In [5]:
!pip install ale-py

In [6]:
!pip install moviepy

In [7]:
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [8]:
!pip install wandb

In [ ]:
directory_videos = '/content/Videos/'

## Algoritmo

In [17]:
import gymnasium as gym
from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
import ale_py
from tqdm import tqdm
from gymnasium.wrappers import RecordEpisodeStatistics, RecordVideo
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv  # Import DummyVecEnv
from EnvironmentWrappers.ObsRewardWrapper import ObsRewardWrapper
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.utils import get_linear_fn
import torch as th
import wandb



/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


## Configurazione di Wandb

In [15]:
!wandb login

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: frank581-fgz (frankzamma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [26]:
# Inizializzazione di wandb
wandb.init(
    project="QBERT-RL",
    entity = "Q-BertRLTeam",

    config={
        "learning_rate": 0.0003,
        "epochs": 10,
    }
)

In [18]:
class WandbCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(WandbCallback, self).__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self) -> bool:
        """Viene chiamato ad ogni step, logga i dati degli episodi."""

        for info in self.locals["infos"]:
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])

        if len(self.episode_rewards) > 0:
            wandb.log({
                "rollout/ep_rew_mean": sum(self.episode_rewards) / len(self.episode_rewards),
                "rollout/ep_len_mean": sum(self.episode_lengths) / len(self.episode_lengths),
                "train/loss": self.model.logger.name_to_value.get("train/loss", 0),
                "train/policy_gradient_loss": self.model.logger.name_to_value.get("train/policy_gradient_loss", 0),
                "train/value_loss": self.model.logger.name_to_value.get("train/value_loss", 0),
            })

        return True

## Training

In [20]:
class OurRewardWrapper(gym.RewardWrapper):
  def reward(self, reward):

    reward =  2 * (reward - (-10)) / (4 - (-10)) - 1

    #print(reward)

    return round(reward, 2)

In [11]:
gym.register_envs(ale_py)

def make_env(env_id):
    def _init():
        env = gym.make(env_id)
        env = ObsRewardWrapper(env)
        # env = OurRewardWrapper(env)
        return env
    return _init

In [42]:
policy_kwargs = dict(activation_fn=th.nn.ReLU,
                      net_arch=dict(pi=[256, 512, 256, 128, 64], vi=[256, 512, 256, 128, 64]))

In [69]:
lr_schedule = get_linear_fn(start=0.001, end=0.00001, end_fraction=0.1)

num_envs = 16
envs = DummyVecEnv([make_env("ALE/Qbert-ram-v5") for _ in range(num_envs)])

ent_coef_start = 1.0
total_timesteps = 1_000_000
timesteps_per_update = 10_000
lambda_decay = 0.001

tensorboard_log_dir = "./tensorboard_logs/"
model = A2C(
    "MlpPolicy",
    envs,
    verbose=1,
    n_steps= 15,
    learning_rate=lr_schedule,
    policy_kwargs=policy_kwargs
)

Using cuda device


In [58]:
import numpy as np

def get_ent_coef(t, ent_coef_start, lambda_decay):
    return ent_coef_start * np.exp(-lambda_decay * t)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [70]:
eval_env =  DummyVecEnv([make_env("ALE/Qbert-ram-v5") for _ in range(1)])
eval_callback = EvalCallback(eval_env, best_model_save_path="./logs/",
                             log_path="./logs/", eval_freq=250,
                             deterministic=True, render=False)
wandb_callback = WandbCallback()

for t in range(0, total_timesteps, timesteps_per_update):
  ent_coef = get_ent_coef(t, ent_coef_start, lambda_decay)
  model.ent_coef = ent_coef
  print("Coefficiente di entropia:" + str(ent_coef) + "t:" + str(t))
  if ent_coef == 0:
    break
  model.learn(total_timesteps=timesteps_per_update,callback=[eval_callback, wandb_callback])



mean_reward, std_reward = evaluate_policy(model, envs, n_eval_episodes=100)
print(f"Ricompensa media: {mean_reward:.2f}, devi#azione standard: {std_reward:.2f}")

Coefficiente di entropia:1.0t:0
Eval num_timesteps=4000, episode_reward=-3.00 +/- 6.00
Episode length: 1038.00 +/- 0.00
------------------------------------
| eval/                 |          |
|    mean_ep_length     | 1.04e+03 |
|    mean_reward        | -3       |
| time/                 |          |
|    total_timesteps    | 4000     |
| train/                |          |
|    entropy_loss       | -0.247   |
|    explained_variance | 0.388    |
|    learning_rate      | 1e-05    |
|    n_updates          | 16       |
|    policy_loss        | -0.33    |
|    value_loss         | 16.8     |
------------------------------------
New best mean reward!
Eval num_timesteps=8000, episode_reward=0.00 +/- 0.00
Episode length: 1038.00 +/- 0.00
------------------------------------
| eval/                 |          |
|    mean_ep_length     | 1.04e+03 |
|    mean_reward        | 0        |
| time/                 |          |
|    total_timesteps    | 8000     |
| train/                |      

In [78]:
mean_reward, std_reward = evaluate_policy(model, envs, n_eval_episodes=100)
print(f"Ricompensa media: {mean_reward:.2f}, devi#azione standard: {std_reward:.2f}")

Ricompensa media: 0.00, devi#azione standard: 0.00


## Registrazione episodi

In [79]:
env = gym.make("ALE/Qbert-ram-v5", render_mode="rgb_array")
env = ObsRewardWrapper(env)

In [80]:
# DummyVecEnv per compatibilità con Stable-Baselines3
env = DummyVecEnv([lambda: env])

In [81]:
# Registra video
video_folder = "./videos/"
env = VecVideoRecorder(
    env,               # Ambiente
    video_folder,     # Cartella per salvare i video
    record_video_trigger=lambda x: x % 1000 == 0,  # Registra ogni 1000 passi
    video_length=100000 # Durata massima del video in passi
)

In [82]:
# Resetta l'ambiente per registrare un episodio
obs = env.reset()

# Registra 3 episodi
for episode in range(3):
    obs = env.reset()
    for _ in range(100000):  # Durata massima dell'episodio
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, dones, info = env.step(action)
        if dones[0]:  # L'episodio è terminato
            break

env.close()  # Salva il video

Moviepy - Building video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-100000.mp4.
Moviepy - Writing video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-100000.mp4



Moviepy - Done !
Moviepy - video ready /content/Q-Bert_RL/videos/rl-video-step-0-to-step-100000.mp4


Moviepy - Building video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-100000.mp4.
Moviepy - Writing video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-100000.mp4



Moviepy - Done !
Moviepy - video ready /content/Q-Bert_RL/videos/rl-video-step-0-to-step-100000.mp4


In [83]:
model.save("model_A2C")

In [84]:
model.load("logs/best_model")